# PidginMed-ASR — Clean Statistical Significance Analysis

In [1]:
# Run once in Colab if needed
%pip -q install jiwer statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 21.3 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from statsmodels.stats.contingency_tables import mcnemar
from jiwer import wer, cer

BASE_DIR = Path('/content/drive/MyDrive/anon/pidgindataset2026October')
INPUT_DIR = BASE_DIR / 'outputRecent'
OUTPUT_DIR = BASE_DIR / 'Models_Results' / 'statistical_analysis_clean'

FILES = {
    'whisper': INPUT_DIR / 'Whisper_test_predictions.csv',
    'canary': INPUT_DIR / 'Canary_test_predictions.csv',
    'medasr': INPUT_DIR / 'MedASR_test_predictions.csv',
}
# Optional source of acoustic features if missing from predictions:
FEATURES_CSV = BASE_DIR / 'metadata_version2.0.csv'
STATISTICAL_SUBSET = 'medical'  # 'medical' (2217) or 'full' (2245)
N_BOOTSTRAP = 1000
RANDOM_SEED = 42
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, path in FILES.items():
    print(name, 'FOUND' if path.is_file() else 'MISSING', path)

## Load and validate original predictions

In [4]:
FEATURE_COLS = ['audio_length', 'speech_rate', 'signal_to_noise_ratio',
                'silence_ratio', 'english_ratio', 'pidgin_ratio', 'cmi']

def load_prediction(model, path):
    if not path.is_file():
        raise FileNotFoundError(f'{model}: {path}')
    df = pd.read_csv(path, dtype={'sample_id': 'string'})
    essential = {'sample_id', 'reference', 'prediction', 'medical_exact_correct'}
    missing = essential.difference(df.columns)
    if missing:
        raise ValueError(f'{model}: missing columns {sorted(missing)}')
    if df['sample_id'].isna().any() or df['sample_id'].duplicated().any():
        raise ValueError(f'{model}: missing or duplicate sample_id')
    if df['reference'].isna().any():
        raise ValueError(f'{model}: missing reference')
    df['prediction'] = df['prediction'].fillna('').astype(str)
    df['reference'] = df['reference'].astype(str)
    df['medical_exact_correct'] = pd.to_numeric(df['medical_exact_correct'], errors='coerce')
    present = df['medical_exact_correct'].dropna()
    if not present.isin([0, 1]).all():
        raise ValueError(f'{model}: medical_exact_correct must contain only 0, 1, NaN')
    if 'medical_num_reference' in df:
        eligible = pd.to_numeric(df['medical_num_reference'], errors='coerce').gt(0)
        if not eligible.equals(df['medical_exact_correct'].notna()):
            raise ValueError(f'{model}: medical eligibility differs from exact-match non-NaN rows')
    return df

models = {name: load_prediction(name, path) for name, path in FILES.items()}
for name, df in models.items():
    print(f'{name}: full={len(df)}, medical={df.medical_exact_correct.notna().sum()}')


whisper: full=2245, medical=2217
canary: full=2245, medical=2217
medasr: full=2245, medical=2217


In [5]:
# Join by sample_id, never row index or filename. Preserve reference from Whisper.
base = models['whisper'].copy()
base = base.rename(columns={'prediction':'whisper_prediction',
                            'medical_exact_correct':'whisper_medical_exact'})
keep = ['sample_id','reference','whisper_prediction','whisper_medical_exact']
if 'audio_path' in base.columns: keep.append('audio_path')
if 'audio_filepath' in base.columns: keep.append('audio_filepath')
for col in FEATURE_COLS:
    if col in base.columns: keep.append(col)
aligned = base[keep].copy()
for name in ('canary', 'medasr'):
    rhs = models[name][['sample_id','reference','prediction','medical_exact_correct']].copy()
    rhs = rhs.rename(columns={'reference':f'{name}_reference',
                              'prediction':f'{name}_prediction',
                              'medical_exact_correct':f'{name}_medical_exact'})
    aligned = aligned.merge(rhs, on='sample_id', how='inner', validate='one_to_one')
    if not aligned['reference'].equals(aligned[f'{name}_reference']):
        bad = aligned.loc[aligned['reference'] != aligned[f'{name}_reference'], 'sample_id']
        raise ValueError(f'Reference mismatch for {name}: {bad.head().tolist()}')
    aligned = aligned.drop(columns=f'{name}_reference')

if len(aligned) != len(models['whisper']) or any(len(aligned) != len(x) for x in models.values()):
    raise ValueError('Incomplete sample_id overlap: cannot silently drop model samples')
med_cols = [f'{x}_medical_exact' for x in ('whisper','canary','medasr')]
mask = aligned[med_cols].notna()
if not mask.eq(mask.iloc[:,0],axis=0).all().all():
    raise ValueError('Different medical-eligibility membership across models')
medical_aligned = aligned.loc[mask.all(axis=1)].copy()
for col in med_cols: medical_aligned[col] = medical_aligned[col].astype(int)
print('Full aligned:',len(aligned),'Medical aligned:',len(medical_aligned))
print('Exact-match accuracy (medical subset):')
print(medical_aligned[med_cols].mean().round(6))
aligned.to_csv(OUTPUT_DIR/'aligned_full_test.csv',index=False)
medical_aligned.to_csv(OUTPUT_DIR/'aligned_medical_test.csv',index=False)


Full aligned: 2245 Medical aligned: 2217
Exact-match accuracy (medical subset):
whisper_medical_exact    0.330627
canary_medical_exact     0.158773
medasr_medical_exact     0.117727
dtype: float64


## Exact McNemar tests — same medical exact-match definition as Table 4

In [6]:
comparisons = [('whisper','medasr'),('whisper','canary'),('medasr','canary')]
rows=[]
for a_name,b_name in comparisons:
    a=medical_aligned[f'{a_name}_medical_exact'].eq(1)
    b=medical_aligned[f'{b_name}_medical_exact'].eq(1)
    both=int((a&b).sum()); a_only=int((a&~b).sum())
    b_only=int((~a&b).sum()); neither=int((~a&~b).sum())
    result=mcnemar([[both,a_only],[b_only,neither]], exact=True)
    rows.append(dict(model_a=a_name.title(),model_b=b_name.title(),
        both_correct=both,a_correct_b_wrong=a_only,a_wrong_b_correct=b_only,
        both_wrong=neither,statistic=float(result.statistic),
        p_value=float(result.pvalue),n=len(medical_aligned)))
mcnemar_results=pd.DataFrame(rows)
assert (mcnemar_results[['both_correct','a_correct_b_wrong','a_wrong_b_correct','both_wrong']]
        .sum(axis=1) == len(medical_aligned)).all()
for name in ('whisper','canary','medasr'):
    total=int(medical_aligned[f'{name}_medical_exact'].sum())
    for row in rows:
        if row['model_a'].lower()==name:
            assert total==row['both_correct']+row['a_correct_b_wrong']
        if row['model_b'].lower()==name:
            assert total==row['both_correct']+row['a_wrong_b_correct']
mcnemar_results.to_csv(OUTPUT_DIR/'mcnemar_exact.csv',index=False)
display(mcnemar_results)


,model_a,model_b,both_correct,a_correct_b_wrong,a_wrong_b_correct,both_wrong,statistic,p_value,n
0,Whisper,Medasr,214,519,47,1437,47.0,1.190276e-101,2217
1,Whisper,Canary,278,455,74,1410,74.0,6.622842e-68,2217
2,Medasr,Canary,138,123,214,1742,123.0,8.169014e-07,2217


## Paired utterance bootstrap (WER and CER)

Bootstrap resamples **aligned utterance indices jointly** for both systems, calculates corpus-level WER/CER for each resample, and reports the 95% percentile interval for **Model A − Model B**. Avoid interpreting a percentile CI as a valid `p < 0.001` test by itself; this notebook intentionally reports **CIs, not bootstrap p-values**. With 1,000 draws, bootstrap tail estimates would have limited resolution anyway. For publication, specify resample count, seed, resampling unit, metric, and sample population.

In [7]:
selected = medical_aligned if STATISTICAL_SUBSET == 'medical' else aligned
if STATISTICAL_SUBSET not in {'medical','full'}: raise ValueError(STATISTICAL_SUBSET)
refs=selected['reference'].fillna('').astype(str).tolist()
models_pred={name:selected[f'{name}_prediction'].fillna('').astype(str).tolist()
             for name in ('whisper','canary','medasr')}
n=len(refs)
if n == 0: raise ValueError('Empty evaluation subset')
rng=np.random.default_rng(RANDOM_SEED)
indices=rng.integers(0,n,size=(N_BOOTSTRAP,n),endpoint=False)
bootstrap_rows=[]
for metric_name, metric_fn in [('wer',wer),('cer',cer)]:
    # share resampled indices across systems and comparisons
    for a_name,b_name in comparisons:
        a=models_pred[a_name]; b=models_pred[b_name]
        observed_a=float(metric_fn(refs,a)); observed_b=float(metric_fn(refs,b))
        diffs=np.empty(N_BOOTSTRAP,dtype=float)
        for i,ix in enumerate(indices):
            rr=[refs[j] for j in ix]
            aa=[a[j] for j in ix]
            bb=[b[j] for j in ix]
            diffs[i]=metric_fn(rr,aa)-metric_fn(rr,bb)
        lo,hi=np.percentile(diffs,[2.5,97.5])
        bootstrap_rows.append(dict(model_a=a_name.title(),model_b=b_name.title(),
            metric=metric_name,n=n,subset=STATISTICAL_SUBSET,
            n_bootstrap=N_BOOTSTRAP,seed=RANDOM_SEED,
            metric_a=observed_a,metric_b=observed_b,
            difference_a_minus_b=observed_a-observed_b,
            ci_95_low=float(lo),ci_95_high=float(hi),
            ci_excludes_zero=bool(lo>0 or hi<0)))
bootstrap_results=pd.DataFrame(bootstrap_rows)
bootstrap_results.to_csv(OUTPUT_DIR/'bootstrap_paired_wer_cer.csv',index=False)
display(bootstrap_results)


,model_a,model_b,metric,n,subset,n_bootstrap,seed,metric_a,metric_b,difference_a_minus_b,ci_95_low,ci_95_high,ci_excludes_zero
0,Whisper,Medasr,wer,2217,medical,1000,42,0.223536,0.303022,-0.079486,-0.090304,-0.067027,True
1,Whisper,Canary,wer,2217,medical,1000,42,0.223536,0.268668,-0.045132,-0.062110,-0.026518,True
2,Medasr,Canary,wer,2217,medical,1000,42,0.303022,0.268668,0.034355,0.020809,0.047577,True
3,Whisper,Medasr,cer,2217,medical,1000,42,0.176966,0.148518,0.028448,0.018971,0.039513,True
4,Whisper,Canary,cer,2217,medical,1000,42,0.176966,0.209710,-0.032745,-0.049831,-0.014052,True
5,Medasr,Canary,cer,2217,medical,1000,42,0.148518,0.209710,-0.061192,-0.075395,-0.047599,True


## Spearman rank correlations

Uses saved acoustic/linguistic features, matching by `sample_id` if external metadata is needed. Correlates **per-utterance** WER/CER/mWER columns from original model outputs (not corpus aggregate metrics). Each result includes its valid-pair sample count. Correlations are observational, and many feature–metric tests can lead to multiple-comparison concerns.

In [8]:
# Features in individual model prediction files take precedence;
# optionally fill missing features by exact sample ID from metadata.
feature_source=None
if FEATURES_CSV.is_file():
    feature_source=pd.read_csv(FEATURES_CSV,dtype={'sample_id':'string'})
    if feature_source['sample_id'].isna().any() or feature_source['sample_id'].duplicated().any():
        raise ValueError('Feature metadata has missing/duplicate sample_id')
    feature_source=feature_source.set_index('sample_id')

spearman_rows=[]
for model_name, df0 in models.items():
    df=df0.copy().set_index('sample_id')
    if feature_source is not None:
        for feature in FEATURE_COLS:
            if feature not in feature_source.columns: continue
            if feature not in df.columns: df[feature]=np.nan
            df[feature]=df[feature].fillna(feature_source[feature])
    if STATISTICAL_SUBSET=='medical':
        df=df.loc[df.index.isin(medical_aligned['sample_id'])]
    for feature in FEATURE_COLS:
        if feature not in df.columns: continue
        for metric in ('wer','cer','mwer'):
            if metric not in df.columns: continue
            dat=df[[feature,metric]].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).dropna()
            if len(dat)<3 or dat[feature].nunique()<2 or dat[metric].nunique()<2:continue
            rho,p=spearmanr(dat[feature],dat[metric])
            spearman_rows.append(dict(model=model_name,feature=feature,metric=metric,
                                      rho=float(rho),p_value=float(p),n=len(dat),subset=STATISTICAL_SUBSET))
spearman_results=pd.DataFrame(spearman_rows,columns=['model','feature','metric','rho','p_value','n','subset'])
spearman_results.to_csv(OUTPUT_DIR/'spearman_correlations.csv',index=False)
display(spearman_results.sort_values(['model','metric','feature']))
print('Spearman rows:',len(spearman_results))


,model,feature,metric,rho,p_value,n,subset
13,canary,audio_length,cer,-0.022959,2.799024e-01,2217,medical
19,canary,signal_to_noise_ratio,cer,0.178595,2.399377e-17,2217,medical
22,canary,silence_ratio,cer,0.212171,5.545398e-24,2217,medical
16,canary,speech_rate,cer,-0.093494,1.036884e-05,2217,medical
14,canary,audio_length,mwer,-0.008636,6.844564e-01,2217,medical
20,canary,signal_to_noise_ratio,mwer,0.145946,5.020496e-12,2217,medical
23,canary,silence_ratio,mwer,0.157333,9.312315e-14,2217,medical
17,canary,speech_rate,mwer,-0.157142,9.981308e-14,2217,medical
12,canary,audio_length,wer,-0.034058,1.088951e-01,2217,medical
18,canary,signal_to_noise_ratio,wer,0.261079,7.149125e-36,2217,medical


Spearman rows: 36


In [9]:
print('Saved to:',OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.glob('*.csv')):
    print(' ',p.name)
print('Exact correctness totals:', {name:int(medical_aligned[f'{name}_medical_exact'].sum())
                                    for name in ('whisper','canary','medasr')})

Saved to: /content/drive/MyDrive/Brock_University/pidgindataset2026October/Models_Results/statistical_analysis_clean
  aligned_full_test.csv
  aligned_medical_test.csv
  bootstrap_paired_wer_cer.csv
  mcnemar_exact.csv
  spearman_correlations.csv
Exact correctness totals: {'whisper': 733, 'canary': 352, 'medasr': 261}
